In [ ]:
import os
import logging
import argparse
import numpy as np
import pandas as pd
from datetime import timedelta, datetime
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import sys
sys.argv = ['']

In [ ]:
# changes
prediction_period = 365 # 365, 730, 1095
tab_size = "/ckd_processed_tab_data_100.csv"
tab_path =  "./../../../commonfilesharePHI/slee/ckd-optum" + tab_size
years = str(round(prediction_period/365))
# target_label_col=f'label_ckd_{years}_year_future' # note

In [ ]:
def parse_args():
    parser = argparse.ArgumentParser(description="CKD classification (n-year future window) and time-to-event training with DeepSurv models on tabular data.")
    # Removed --embedding-root as we use tabular CSV directly
    parser.add_argument("--tabular-data-file", type=str, default=tab_path, help="Path to the processed tabular CKD data CSV file.")
    parser.add_argument("--window-size", type=int, default=10, help="Sequence window size.")
    parser.add_argument("--embed-dim", type=int, default=None, help="Dimensionality of features (will be auto-detected if None).") # Modified
    parser.add_argument("--epochs", type=int, default=50, help="Number of epochs per model.")
    parser.add_argument("--batch-size", type=int, default=64, help="Batch size.")
    parser.add_argument("--lr", type=float, default=5e-5, help="Learning rate.")
    parser.add_argument("--patience", type=int, default=5, help="Early stopping patience.")
    parser.add_argument("--scheduler-patience", type=int, default=2, help="Patience for scheduler LR reduction.")
    # Removed --metadata-file as it's combined into --tabular-data-file
    parser.add_argument("--random-seed", type=int, default=42, help="Random seed.")
    parser.add_argument("--hidden-dim", type=int, default=128, help="Hidden dimension for models (RNN, LSTM, Transformer, MLP, TCN).")
    parser.add_argument("--num-layers", type=int, default=4, help="Number of layers for models.")
    parser.add_argument("--rnn-dropout", type=float, default=0.2, help="Dropout in RNN/LSTM.")
    parser.add_argument("--rnn-bidir", action="store_true", help="Use bidirectional RNN/LSTM if set.")
    parser.add_argument("--transformer-nhead", type=int, default=4, help="Number of heads in Transformer encoder.")
    parser.add_argument("--transformer-dim-feedforward", type=int, default=256, help="Feedforward dim in Transformer layers.")
    parser.add_argument("--transformer-dropout", type=float, default=0.2, help="Dropout in Transformer layers.")
    parser.add_argument("--max-patients", type=int, default=None, help="If set, only use data for up to this many patients.")
    parser.add_argument("--output-model-prefix", type=str, default=f"best_tab_model_{years}yr_future", help="Filename prefix for saved models.") # Updated prefix
    parser.add_argument("--log-tte", action="store_true", help="Apply log transformation to time-to-event targets") # Kept for consistency if TTE values are used
    parser.add_argument("--num-workers", type=int, default=0, help="Number of DataLoader workers (set to 0 for Windows or debugging).") # Default to 0 for broader compatibility
    parser.add_argument("--prediction-horizon-days", type=int, default=prediction_period, help="Number of days into the future to check for an event for label generation.")
    return parser.parse_args()

In [ ]:
args = parse_args()
metadata = pd.read_csv(args.tabular_data_file, parse_dates=["EventDate"])

In [ ]:
metadata.head()

In [ ]:
def clean_ckd_stage(value):
    try:
        return int(value)
    except ValueError:
        if isinstance(value, str) and value[0].isdigit():
            return int(value[0])
        else:
            return np.nan
            
def filter_patients_by_ckd_stage(df, ckd_stage_col, patient_id_col='PatientID'):
    initial_patients = df[patient_id_col].nunique()
    # Filter for visits where CKD stage is 3 or higher
    df_at_or_above_stage_3 = df[df[ckd_stage_col] >= 3]
    # Get unique PatientIDs from this filtered DataFrame
    patient_ids_to_keep = set(df_at_or_above_stage_3[patient_id_col].unique())
    
    patients_removed = initial_patients - len(patient_ids_to_keep)

    return patient_ids_to_keep        

def find_CKD_stage_progression(df):
    df_sorted = df.sort_values(by=['PatientID', 'EventDate_dt'])
    
    # difference in CKD_stage for each patient
    df_sorted['stage_diff'] = df_sorted.groupby('PatientID')['CKD_stage_clean'].diff()

    # filter where the stage difference is positive (i.e., increased)
    df_increased = df_sorted[df_sorted['stage_diff'] > 0].copy()

    # retreive previous CKD_stage for context
    df_increased['previous_CKD_stage'] = df_sorted.groupby('PatientID')['CKD_stage_clean'].shift(1)
    
    # rename relevant columns
    result = df_increased[['PatientID', 'EventDate_dt', 'previous_CKD_stage', 'CKD_stage_clean']]
    result.rename(columns={'CKD_stage_clean': 'new_CKD_stage'}, inplace=True)
    
    return result

def unique_patient_ckd_counts(df):
    # Select only the necessary columns and drop duplicate rows based on PatientID
    # to ensure each patient is counted only once for their CKD stage.
    unique_patients_ckd = df[['PatientID', 'CKD_stage_clean']].drop_duplicates(subset=['PatientID'])

    # Count the occurrences of each CKD stage among these unique patients
    ckd_stage_counts = unique_patients_ckd['CKD_stage_clean'].value_counts()

    return ckd_stage_counts.sort_index()

In [ ]:
metadata['CKD_stage_clean'] = metadata['CKD_stage'].apply(clean_ckd_stage)
metadata = metadata.sort_values(by=['PatientID', 'EventDate'])
metadata['CKD_stage_clean'] = metadata.groupby('PatientID')['CKD_stage_clean'].bfill().ffill()
metadata = metadata.dropna(subset=['CKD_stage_clean'])
metadata['CKD_stage_clean'] = metadata['CKD_stage_clean'].astype(int)
metadata['label'] = metadata['CKD_stage_clean'].apply(lambda x: 1 if x >= 4 else 0)

In [ ]:
metadata['EventDate_dt'] = pd.to_datetime(metadata['EventDate'], errors='coerce')

In [ ]:
increased_stages_df = find_CKD_stage_progression(metadata)
increased_stages_df.shape

In [ ]:
increased_stages_df.head()

In [ ]:
# total number of patients
npatients = len(metadata['PatientID'].unique())
print(npatients)

In [ ]:
df = metadata
df.shape
# check nan rows
# nan_mean = df.isnull().mean()
# nan_mean

In [ ]:
df_patients = filter_patients_by_ckd_stage(metadata, 'CKD_stage_clean')
df = df[df["PatientID"].isin(df_patients)].copy()

In [ ]:
# total number of patients after filtering
npatients = len(df['PatientID'].unique())
print(npatients)

In [ ]:
df.shape

In [ ]:
increased_stages_df = find_CKD_stage_progression(df)
increased_stages_df

In [ ]:
increased_stages_df.shape

In [ ]:
increased_stages_df['new_CKD_stage'].hist()

In [ ]:
# compare progression to true values (pre filtering)
df1 = pd.read_csv("./365day_future_prediction_outputs_50/LSTM_365DayFutureTarget_detailed_outputs.csv")
df1.head()

In [ ]:
df1.shape

In [ ]:
len(df1['PatientID'].unique())

In [ ]:
# wip
def get_yearly_progression_counts(df):

    # Ensure 'EventDate_dt' is in datetime format
    df['EventDate_dt'] = pd.to_datetime(df['EventDate_dt'])

    # 1. Identify all CKD_stage_clean increase events
    df_sorted = df.sort_values(by=['PatientID', 'EventDate_dt'])
    df_sorted['stage_diff'] = df_sorted.groupby('PatientID')['CKD_stage_clean'].diff()
    
    # Filter for rows where the stage difference is positive (i.e., increased)
    # We only need 'PatientID' and 'EventDate_dt' for counting purposes
    progression_events = df_sorted[df_sorted['stage_diff'] > 0][['PatientID', 'EventDate_dt']].copy()

    if progression_events.empty:
        print("No CKD_stage_clean progression events found in the dataset.")
        return pd.DataFrame(columns=['Year', 'Num_Patients_Progressed'])

    # 2. Extract the year from the event time
    progression_events['Year'] = progression_events['EventDate_dt'].dt.year

    # 3. Count unique patients who progressed each year
    # Group by year and count unique patient IDs
    yearly_progression_counts = progression_events.groupby('Year')['PatientID'].nunique().reset_index()
    yearly_progression_counts.rename(columns={'PatientID': 'Num_Patients_Progressed'}, inplace=True)
    
    # Get valid years from the 'time of the event' column, dropping any NaNs
    valid_years = df['EventDate_dt'].dt.year.dropna()

    if valid_years.empty:
        # If there are no valid years (e.g., df is empty, or all dates are invalid),
        # return a DataFrame with a default year and 0 progressions.
        # This prevents the 'float object cannot be interpreted as an integer' error
        # when trying to create a range from min_year/max_year that would be NaN.
        current_year = datetime.datetime.now().year
        return pd.DataFrame({'Year': [current_year], 'Num_Patients_Progressed': [0]})
    
    # Ensure min_year and max_year are integers
    min_year = int(valid_years.min())
    max_year = int(valid_years.max())
    all_years = pd.DataFrame({'Year': range(min_year, max_year + 1)})
    
    yearly_progression_counts = pd.merge(all_years, yearly_progression_counts, on='Year', how='left')
    yearly_progression_counts['Num_Patients_Progressed'].fillna(0, inplace=True)
    yearly_progression_counts['Num_Patients_Progressed'] = yearly_progression_counts['Num_Patients_Progressed'].astype(int)

    return yearly_progression_counts

In [ ]:
# wip
progression_by_years = get_yearly_progression_counts(df)
progression_by_years.sort_values(by='Num_Patients_Progressed', ascending=False).head(10)

In [ ]:
# leave out
# find total number of events from each year
patients_by_year = df.groupby(df['EventDate_dt'].dt.year).size().reset_index(name='event_count').sort_values(by="event_count", ascending=False)
ax = patients_by_year[:20].plot(kind='bar', x='EventDate_dt', grid=True)
ax.tick_params(axis='x', labelrotation=45)
ax.set_title('Event Count for the Top 20 Years')
ax.set_xlabel("year")
ax.set_ylabel("event count")

In [ ]:
# leave out
# chronological
patients_by_year2 = df.groupby(df['EventDate_dt'].dt.year).size().reset_index(name='event_count').sort_values(by="EventDate_dt", ascending=False)
ax = patients_by_year2[:20].plot(kind='bar', x='EventDate_dt', grid=True)
ax.tick_params(axis='x', labelrotation=45)
# ehr adopted in 2013, expanding over time
ax.set_title('Chronological Event Count by Year')
ax.set_xlabel("year")
ax.set_ylabel("event count")

In [ ]:
events_per_patient = df['PatientID'].value_counts()
events_per_patient.head()

In [ ]:
events_by_stage = df.groupby(df['CKD_stage_clean']).size().reset_index(name='event_count')
events_by_stage

In [ ]:
# pre filtering
patients_by_stage = unique_patient_ckd_counts(metadata)
patients_by_stage
ax = patients_by_stage[:20].plot(kind='bar', x='CKD_stage', grid=True)
ax.tick_params(axis='x', labelrotation=0)
ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.set_title(' Patient Count by CKD Stage')
ax.set_xlabel("year")
ax.set_ylabel("patient count")

In [ ]:
# post filtering
patients_by_stage = unique_patient_ckd_counts(df)
patients_by_stage
ax = patients_by_stage[:20].plot(kind='bar', x='CKD_stage', grid=True)
ax.tick_params(axis='x', labelrotation=0)
ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.set_title(' Patient Count by CKD Stage')
ax.set_xlabel("year")
ax.set_ylabel("patient count")